In [0]:
df = ( spark.readStream.format("cloudFiles")
                     .option("cloudFiles.format" , 'csv')
                     .option("cloudFiles.schemaLocation" , '/Volumes/ashlamba/bronze/autovol/Orders/Schema/')
                     .option("cloudFiles.schemaEvolutionMode" , "addNewColumns") #addNewColumns
                     .load("/Volumes/ashlamba/bronze/autovol/Orders/Raw/")
)

In [0]:
df.writeStream.format("delta")\
              .outputMode("append")\
              .option("checkpointLocation" , '/Volumes/ashlamba/bronze/autovol/Orders/Checkpoint/')\
              .trigger(availableNow = True)\
              .option("mergeSchema" , True)\
              .start('/Volumes/ashlamba/bronze/autovol/Orders/Data/')

In [0]:
from pyspark.sql.functions import col,expr
count = spark.read.format("delta").load("/Volumes/ashlamba/bronze/autovol/Orders/Data/")
count.select(expr("*")).display()

In [0]:
from pyspark.sql.types import DecimalType , StringType, StructType , StructField 
from pyspark.sql.functions import from_json , col

schema = StructType(
    [
        StructField("discount",DecimalType(5,2)),
        StructField("payment_method",StringType()),
        StructField("_file_path",StringType())
    ]
)

count1 = count.withColumn("struct_rescued" , from_json(col("_rescued_data"),schema))
count2 = count1.withColumns(
    {
        "discount" : col("struct_rescued")['discount'],
        "payment_method" : col("struct_rescued")['payment_method'],
        "file_path" : col("struct_rescued")['_file_path']
    }
)

count2.display()

In [0]:
count2.select("struct_rescued.*").display()